# 03 Build matched canine analysis matrix

Purpose: merge GSE238110 log2-CPM expression with matched DOG2 clinical metadata and create analysis-ready matrices.

In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)

In [ ]:
expr_path = PROCESSED_DIR / "GSE238110_log2cpm_detection20.csv"
clinical_path = PROCESSED_DIR / "DOG2_clinical_matched_to_GSE238110.csv"

expr = pd.read_csv(expr_path, index_col=0)
clinical = pd.read_csv(clinical_path)

print("Expression shape genes x samples:", expr.shape)
print("Clinical shape:", clinical.shape)
display(expr.iloc[:5, :5])
display(clinical.head())

In [ ]:
sample_col = "original_sample_column"
available_samples = [s for s in clinical[sample_col].astype(str) if s in expr.columns]

expr_matched = expr[available_samples].T
expr_matched.index.name = "sample_id"
clinical_matched = clinical.set_index(sample_col).loc[available_samples].copy()

print("Matched expression shape samples x genes:", expr_matched.shape)
print("Matched clinical shape:", clinical_matched.shape)
display(expr_matched.iloc[:5, :5])
display(clinical_matched.head())

In [ ]:
gene_variance = expr_matched.var(axis=0).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.hist(gene_variance, bins=60)
plt.xlabel("Gene variance")
plt.ylabel("Number of genes")
plt.title("Gene variance after log2-CPM normalization")
plt.show()

top_n = 5000
top_variable_genes = gene_variance.head(top_n).index.tolist()
expr_top = expr_matched[top_variable_genes].copy()

print("Top variable matrix shape:", expr_top.shape)

In [ ]:
endpoint_summary = []
for col in ["metastasis_event", "os_time", "os_event", "dfi_time", "dfi_event"]:
    if col in clinical_matched.columns:
        endpoint_summary.append({
            "column": col,
            "non_missing": int(clinical_matched[col].notna().sum()),
            "unique_values": int(clinical_matched[col].nunique(dropna=True))
        })

display(pd.DataFrame(endpoint_summary))

In [ ]:
analysis = clinical_matched.join(expr_top, how="inner")
analysis.to_csv(PROCESSED_DIR / "GSE238110_DOG2_analysis_top5000var.csv")

expr_matched.to_csv(PROCESSED_DIR / "GSE238110_DOG2_expression_log2cpm_matched_allgenes.csv")
expr_top.to_csv(PROCESSED_DIR / "GSE238110_DOG2_expression_log2cpm_matched_top5000var.csv")

print("Saved analysis files:")
print(PROCESSED_DIR / "GSE238110_DOG2_analysis_top5000var.csv")
print(PROCESSED_DIR / "GSE238110_DOG2_expression_log2cpm_matched_allgenes.csv")
print(PROCESSED_DIR / "GSE238110_DOG2_expression_log2cpm_matched_top5000var.csv")